## Now that we were able to figure outif the ship was going toward a place (which was hard), all that is left to do is to show if it is going toward some kind of a port.
### This should be easy, we should just be able to take the bearing the ship would need to go to a location, and then compare it to the actual bearing of the ship

# Setup:

In [1]:
DEGREE_THRESHOLD = 15

# current_position = (15, -153) #Near Hawaii going toward North America/(United States)
# direction_vector = (0, 10)

# current_position = (28, -59) # In the Atlantic off the coast of the United States going toward the Gulf of America
# direction_vector = (-10, -2.5)

current_position = (40, 17) # In the middle of the Mediter
direction_vector = (-6, 9)

In [2]:
from ipyleaflet import Map, Marker, AwesomeIcon, Popup, Rectangle
map1 = Map(center=(current_position[0], current_position[1]), zoom=10)

# Add first and last markers
current_pos_marker = Marker(location=(current_position[0], current_position[1]))
map1.add(current_pos_marker)

(dy, dx) = direction_vector



start_lat = current_position[0]
start_lon = current_position[1]

end_lat = start_lat + dy
end_lon = start_lon + dx

from ipyleaflet import Polyline

vector_line = Polyline(
    locations=[(start_lat, start_lon), (end_lat, end_lon)],
    color="red",
    weight=3
)

map1.add(vector_line)

map1

Map(center=[40, 17], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_tex…

## Map of all the important ports/waterways

In [3]:

from ipywidgets import HTML
import math
from actint.tools.utils.important_locations import *

# Create map centered somewhere reasonable and set zoom
map2 = Map(center=(20.0, 0.0), zoom=2)

# Optional helper for text popup at center

from ipyleaflet import CircleMarker

def create_point_on_map(map_obj, latlon, color='blue', fill_opacity=0.15, weight=2, name=None):
    """
    Adds a circular point/marker to the map.
    latlon: tuple of (latitude, longitude)
    """
    point = CircleMarker(
        location=latlon,
        radius=5,                # Size of the point
        color=color,             # Border color
        fill_color=color,        # Interior color
        fill_opacity=fill_opacity,
        weight=weight,           # Border thickness
        label=name               # Hover text (if supported by your widget version)
    )
    
    # Optional: Add a popup if a name is provided
    if name:
        point.popup = HTML(value=f"<b>{name}</b>")

    map_obj.add_layer(point)
    return point
    

for key, value in STRATEGIC_WATERWAYS.items():
    create_point_on_map(map2, value, color='blue', fill_opacity=0.08, weight=1, name=key)

for key, value in MAJOR_PORTS.items():
    create_point_on_map(map2, value, color='red', fill_opacity=0.08, weight=1, name=key)


map2

Map(center=[20.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…

## Utility Functions:

In [4]:
def vector_to_latlon_degrees(lat_diff, lon_diff):
    # Standard atan2 is (y, x). 
    # For navigation: y = lon (East/West), x = lat (North/South)
    # This makes 0 degrees = North
    radians = math.atan2(lon_diff, lat_diff)
    degrees = math.degrees(radians)
    
    return degrees

    return degrees


current_heading = vector_to_latlon_degrees(direction_vector[0], direction_vector[1])

## Find the needed ship heading to get to a location and compare it with the actual heading

In [5]:
from geographiclib.geodesic import Geodesic
geod = Geodesic.WGS84

def ship_going_toward_location(current_position, current_heading, end_position, degree_threshold):
    path = geod.Inverse(current_position[0], current_position[1], end_position[0], end_position[1])
    needed_heading = path['azi1']

    differential_heading = (needed_heading - current_heading + 180) % 360 - 180
    
    if differential_heading > -DEGREE_THRESHOLD and differential_heading < DEGREE_THRESHOLD:
        return path['s12']
    else: 
        return None
    
hit_ports = []
for port in MAJOR_PORTS.items():
    distance = ship_going_toward_location(current_position, current_heading, port[1], DEGREE_THRESHOLD)
    if(distance):
        hit_ports.append((port[0], distance))

hit_waterways = []
for waterway in STRATEGIC_WATERWAYS.items():
    distance = ship_going_toward_location(current_position, current_heading, waterway[1], DEGREE_THRESHOLD)
    if(distance):
        hit_waterways.append((waterway[0], distance))

print(hit_ports)
print(hit_waterways)



[]
[('Suez Canal', 1748114.2431530068), ('Bab el-Mandeb', 3986126.4423535927)]


## Display ship potential ports on map:

In [6]:

def project_point(lat, lon, bearing, distance_km):
    # geographiclib uses meters, so we multiply km by 1000
    # Direct returns a dictionary with 'lat2' and 'lon2'
    result = geod.Direct(lat, lon, bearing, distance_km * 1000)
    return result['lat2'], result['lon2']

def view_approximation(lat, lon, bearing, map_obj, degree_threshold, distance_km=50000):
    # Upper bound line
    points_up = []
    for i in range(1, 51):
        dist = distance_km * (i / 50)
        points_up.append(project_point(lat, lon, bearing + degree_threshold, dist))
    
    line_up = Polyline(
        locations=points_up,
        color="red",
        weight=1
    )
    map_obj.add(line_up)

    # Lower bound line
    points_down = []
    for i in range(1, 51):
        dist = distance_km * (i / 50)
        points_down.append(project_point(lat, lon, bearing - degree_threshold, dist))
    
    line_down = Polyline(
        locations=points_down,
        color="blue",
        weight=1 # Assuming you wanted the second line thinner or default
    )
    map_obj.add(line_down)



map3 = Map(center=(20.0, 0.0), zoom=2)


# Use a smaller threshold, like 15 degrees
view_approximation(current_position[0], current_position[1], current_heading, map3, DEGREE_THRESHOLD, distance_km=5000)

for key, value in STRATEGIC_WATERWAYS.items():
    create_point_on_map(map3, value, color='blue', fill_opacity=0.08, weight=1, name=key)

for key, value in MAJOR_PORTS.items():
    create_point_on_map(map3, value, color='red', fill_opacity=0.08, weight=1, name=key)


map3

Map(center=[20.0, 0.0], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_out_…